# N-1 Contingency Analysis Notebook
This notebook demonstrates running N-1 analysis on the power system. The N-1 analysis takes an line that is out of service and returns alternative topologies with the alternative line ID, maximum loading among the lines at a certain timestamp, at which line this maximum occurs and the timestamp at which this occurs.

In [1]:
import os
import sys
from pathlib import Path

# Add src to path for imports
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

# Verify we're in the right directory
print(f"Current working directory: {os.getcwd()}")
print(f"Repository root: {REPO_ROOT}")

Current working directory: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky\example
Repository root: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky


Import appropriate packages to run N-1

In [2]:
from power_system_simulation.lv_grid_analytics import LVGridAnalytics
from power_system_simulation.N_minus_1 import InvalidLineOutageError

print("✓ Imports successful")

✓ Imports successful


Import lv grid data set to be tested.

In [3]:
# Define test data paths
TEST_DATA_DIR = REPO_ROOT / "tests" / "big_network" / "big_network" / "input"

print(f"Test data directory: {TEST_DATA_DIR}")
print(f"Directory exists: {TEST_DATA_DIR.exists()}")

# List files in the directory
if TEST_DATA_DIR.exists():
    print("\nFiles in test directory:")
    for file in sorted(TEST_DATA_DIR.iterdir()):
        print(f"  - {file.name}")

Test data directory: c:\Users\Michiel\OneDrive\Documents\GitHub\power-system-simulation-Sparky\tests\big_network\big_network\input
Directory exists: True

Files in test directory:
  - active_power_profile.parquet
  - ev_active_power_profile.parquet
  - input_network_data.json
  - meta_data.json
  - reactive_power_profile.parquet


In [4]:
# Create LVGridAnalytics instance
analytics = LVGridAnalytics(
    grid_path=str(TEST_DATA_DIR / "input_network_data.json"),
    meta_data=str(TEST_DATA_DIR / "meta_data.json"),
    active_load_profile_path=str(TEST_DATA_DIR / "active_power_profile.parquet"),
    reactive_load_profile_path=str(TEST_DATA_DIR / "reactive_power_profile.parquet"),
    ev_profile_path=str(TEST_DATA_DIR / "ev_active_power_profile.parquet"),
)

print("✓ LVGridAnalytics instance created")

✓ LVGridAnalytics instance created


In [5]:
# Validate inputs
analytics.validate_inputs()
print("✓ Inputs validated")

✓ Inputs validated


In [6]:
# Check the grid structure
print("Dataset info:")
print(f"  Lines: {len(analytics._dataset['line'])} lines")
print(f"  Nodes: {len(analytics._dataset['node'])} nodes")
print(f"  Loads: {len(analytics._dataset['sym_load'])} loads")
print(f"\nActive load profile shape: {analytics._active_load_profiles.shape}")
print(f"Reactive load profile shape: {analytics._reactive_load_profiles.shape}")

Dataset info:
  Lines: 807 lines
  Nodes: 802 nodes
  Loads: 400 loads

Active load profile shape: (35040, 400)
Reactive load profile shape: (35040, 400)


# Running the N-1 analysis

In [ ]:
# Run N-1 analysis for line 1204
print("Running N-1 contingency analysis for line 1204...\n")
results_line_1204 = analytics.n_minus_one(outage_line_id=1204)

print(f"Results shape: {results_line_1204.shape}")
print(f"\nColumns: {list(results_line_1204.columns)}")
print(f"\nResults:\n{results_line_1204}")

Running N-1 contingency analysis for line 16...

Results shape: (1, 4)

Columns: ['Alternative_Line_ID', 'Max_Loading', 'Max_Loading_Line_ID', 'Max_Loading_Timestamp']

Results:
   Alternative_Line_ID  Max_Loading  Max_Loading_Line_ID Max_Loading_Timestamp
0                 2004     0.129424                 1310   2025-11-05 02:00:00


In [15]:
# Run N-1 analysis for line 1324
print("Running N-1 contingency analysis for line 1324...\n")
results_line_1324 = analytics.n_minus_one(outage_line_id=1324)

print(f"Results shape: {results_line_1324.shape}")
print(f"\nResults:\n{results_line_1324}")

Running N-1 contingency analysis for line 1324...

Results shape: (2, 4)

Results:
   Alternative_Line_ID  Max_Loading  Max_Loading_Line_ID Max_Loading_Timestamp
0                 2004     0.104716                 1232   2025-11-08 10:45:00
1                 2005     0.113928                 1406   2025-11-05 16:30:00


Test N-1 with invalid line or line that is already out of service.

In [ ]:
# Test with invalid line ID
print("Testing with invalid line ID (999)...\n")
try:
    results_invalid = analytics.n_minus_one(outage_line_id=999)
except InvalidLineOutageError as e:
    print(f"✓ Correctly caught error: {e}")

Testing with invalid line ID (999)...

✓ Correctly caught error: Line ID 2005 is already out of service.


In [17]:
# Test with Line ID that is not connected
print("Testing with out of service line ID (2005)...\n")
try:
    results_invalid = analytics.n_minus_one(outage_line_id=2005)
except InvalidLineOutageError as e:
    print(f"✓ Correctly caught error: {e}")

Testing with out of service line ID (2005)...

✓ Correctly caught error: Line ID 2005 is already out of service.


In [ ]:
# Test with Line ID with no alternatives
print("Testing with Line ID with no alternatives (2003)...\n")
results_invalid = analytics.n_minus_one(outage_line_id=2003)
print(f"Results shape: {results_invalid.shape}")
print(f"\nResults:\n{results_invalid}")

Testing with out of service line ID (1988)...

Results shape: (0, 4)

Results:
Empty DataFrame
Columns: [Alternative_Line_ID, Max_Loading, Max_Loading_Line_ID, Max_Loading_Timestamp]
Index: []


In [ ]:
# Summary
print("\n" + "=" * 60)
print("N-1 CONTINGENCY ANALYSIS SUMMARY")
print("=" * 60)
print("\nLine 1204 Outage:")
print(f"  - Number of alternatives: {len(results_line_1204)}")
if len(results_line_1204) > 0:
    print(
        f"  - Max loading range: "
        f"{results_line_1204['Max_Loading'].min():.4f} - "
        f"{results_line_1204['Max_Loading'].max():.4f}"
    )
    print(f"  - Alternative lines: {sorted(results_line_1204['Alternative_Line_ID'].tolist())}")

print("\nLine 1324 Outage:")
print(f"  - Number of alternatives: {len(results_line_1324)}")
if len(results_line_1324) > 0:
    print(
        f"  - Max loading range: "
        f"{results_line_1324['Max_Loading'].min():.4f} - "
        f"{results_line_1324['Max_Loading'].max():.4f}"
    )
    print(f"  - Alternative lines: {sorted(results_line_1324['Alternative_Line_ID'].tolist())}")

print("\n✓ N-1 analysis completed successfully!")


N-1 CONTINGENCY ANALYSIS SUMMARY

Line 16 Outage:


NameError: name 'results_line_1204' is not defined